# CSIRO Image2Biomass: Dual-Stream DINO + Interval Classification

This notebook implements the winning **1st-Place Solution** architecture:
1. **Dual-Stream Tiling (1000×1000 $\to$ 512×512)**: Centerline split into natural 1:1 square Left and Right views.
2. **Cross-View Multi-Head Self-Attention Interaction**: Interacting tokens across the pasture seam before fusion.
3. **Auxiliary Interval Classification (UEPNet 7-bin)**: Stabilizing regression gradients with discrete count intervals.
4. **Decoupled Training**: 5 independent regression heads + 5 classification heads; soft physical post-processing at test time.
5. **Two-Stage Fine-Tuning**: Stage 1 (freeze backbone) + Stage 2 (end-to-end fine-tuning with differential LR).


In [ ]:
# 1. Environment & Installations
!pip install -q timm

import os
import sys
import glob
import random
import math
import numpy as np
import pandas as pd
from PIL import Image
import cv2
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import StratifiedGroupKFold
from torchvision import transforms
import timm

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
# 2. Configuration & Automatic Path Discovery
def find_data_dir():
    # 1. Check known competition mount locations first
    candidates = [
        '/kaggle/input/competitions/csiro-biomass',
        '/kaggle/input/csiro-biomass',
        '../input/competitions/csiro-biomass',
        '../input/csiro-biomass',
        './data',
        '.'
    ]
    for c in candidates:
        if os.path.exists(os.path.join(c, 'train.csv')):
            return c
    # 2. Walk /kaggle/input if mounted in unexpected subfolder
    if os.path.exists('/kaggle/input'):
        for root, _, files in os.walk('/kaggle/input'):
            if 'train.csv' in files:
                return root
    return '.'

class CFG:
    DATA_DIR = find_data_dir()
    TRAIN_CSV = os.path.join(DATA_DIR, 'train.csv')
    TEST_CSV = os.path.join(DATA_DIR, 'test.csv')
    TRAIN_IMG_DIR = os.path.join(DATA_DIR, 'train') if os.path.exists(os.path.join(DATA_DIR, 'train')) else DATA_DIR
    TEST_IMG_DIR = os.path.join(DATA_DIR, 'test') if os.path.exists(os.path.join(DATA_DIR, 'test')) else DATA_DIR
    
    # Model parameters
    BACKBONE = 'vit_base_patch16_dinov3_qkvb'  # Alternative: 'vit_base_patch14_dinov2'
    IMG_SIZE = 512       # 512x512 per view (1024x1024 effective field)
    FUSION_DIM = 384
    DROPOUT = 0.3
    
    # Training parameters
    STAGE1_EPOCHS = 8    # Frozen backbone (heads + fusion only)
    STAGE2_EPOCHS = 20   # Full end-to-end fine-tuning
    BATCH_SIZE = 8
    LR = 3e-4
    BACKBONE_LR_FACTOR = 0.1  # Differential learning rate for backbone
    WEIGHT_DECAY = 0.05
    MAX_GRAD_NORM = 1.0
    N_FOLDS = 5
    
    # Loss & Auxiliary Interval Classification
    NUM_INTERVALS = 7
    CLS_WEIGHT = 0.3
    CAMERA_SCALE_PROB = 0.2
    USE_TTA = True

    # Targets & Official Metric Weights
    TARGET_ORDER = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
    OFFICIAL_WEIGHTS = [0.1, 0.1, 0.1, 0.2, 0.5]
    IMAGENET_MEAN = [0.485, 0.456, 0.406]
    IMAGENET_STD = [0.229, 0.224, 0.225]

print(f'Discovered DATA_DIR: {CFG.DATA_DIR}')
print(f'TRAIN_CSV: {CFG.TRAIN_CSV} (Exists: {os.path.exists(CFG.TRAIN_CSV)})')
print(f'TEST_CSV:  {CFG.TEST_CSV} (Exists: {os.path.exists(CFG.TEST_CSV)})')
print(f'Backbone: {CFG.BACKBONE} | Image Size: {CFG.IMG_SIZE}x{CFG.IMG_SIZE} | Folds: {CFG.N_FOLDS}')


In [ ]:
# 3. UEPNet Interval Partitioning, Metric, & Soft Post-Processing
BORDERS_DICT = {
    'Dry_Green_g':  [1.6e-05, 13.4232, 27.0782, 45.5236, 79.834, 157.9836],
    'Dry_Dead_g':   [1.6e-05, 6.1407, 13.1192, 23.277, 38.8581, 83.8407],
    'Dry_Clover_g': [1.6e-05, 3.9, 10.5353, 20.6523, 37.5911, 71.7865],
    'GDM_g':        [1.6e-05, 16.5143, 30.507, 49.5585, 81.0, 157.9836],
    'Dry_Total_g':  [1.6e-05, 23.4907, 41.1, 61.1, 96.8288, 185.7],
}

def get_interval_labels(targets_np, target_cols=CFG.TARGET_ORDER):
    labels_cls = np.zeros_like(targets_np, dtype=np.int64)
    for col_idx, col_name in enumerate(target_cols):
        borders = BORDERS_DICT.get(col_name)
        if borders is not None:
            labels_cls[:, col_idx] = np.digitize(targets_np[:, col_idx], borders)
        else:
            labels_cls[:, col_idx] = np.clip(np.digitize(targets_np[:, col_idx], [0, 5, 15, 30, 60, 120]), 0, 6)
    return labels_cls

class WeightedBiomassLoss(nn.Module):
    def __init__(self, loss_weights=CFG.OFFICIAL_WEIGHTS, cls_weight=CFG.CLS_WEIGHT):
        super().__init__()
        self.criterion_reg = nn.SmoothL1Loss()
        self.criterion_cls = nn.CrossEntropyLoss()
        self.cls_weight = cls_weight
        self.weights = loss_weights

    def forward(self, predictions_reg, predictions_cls, targets_reg, targets_cls=None):
        device = targets_reg.device
        w = torch.tensor(self.weights, device=device, dtype=torch.float32)
        
        loss_reg = torch.tensor(0.0, device=device)
        for i in range(5):
            pred_i = predictions_reg[i].squeeze(-1) if isinstance(predictions_reg, list) else predictions_reg[:, i]
            loss_reg += w[i] * self.criterion_reg(pred_i, targets_reg[:, i])

        loss_cls = torch.tensor(0.0, device=device)
        if predictions_cls is not None and targets_cls is not None:
            for i in range(5):
                loss_cls += w[i] * self.criterion_cls(predictions_cls[i], targets_cls[:, i].long())

        total_loss = loss_reg + (self.cls_weight * loss_cls)
        return total_loss, loss_reg, loss_cls

def calculate_competition_r2(y_true, y_pred, weights=CFG.OFFICIAL_WEIGHTS):
    y_true = np.array(y_true, dtype=float).reshape(-1, 5)
    y_pred = np.array(y_pred, dtype=float).reshape(-1, 5)
    w = np.array(weights, dtype=float)
    
    yt = np.log1p(np.maximum(0, y_true))
    yp = np.log1p(np.maximum(0, y_pred))
    
    r2_scores = []
    for i in range(5):
        ss_res = np.sum((yt[:, i] - yp[:, i]) ** 2)
        ss_tot = np.sum((yt[:, i] - np.mean(yt[:, i])) ** 2)
        score = 1.0 - (ss_res / ss_tot) if ss_tot != 0 else (1.0 if ss_res == 0 else 0.0)
        r2_scores.append(score)
    return float(np.sum(w * np.array(r2_scores)))

def soft_physics_postprocess(preds_np):
    preds = np.maximum(preds_np.copy(), 0.0)
    green = preds[:, 0]
    dead = preds[:, 1]
    clover = preds[:, 2] * 0.8
    gdm = preds[:, 3]
    total = preds[:, 4]
    
    dead = np.where(dead > 20.0, dead * 1.1, np.where(dead < 10.0, dead * 0.9, dead))
    gdm_blended = 0.5 * gdm + 0.5 * (green + clover)
    total_blended = 0.5 * total + 0.5 * (green + clover + dead)
    
    return np.maximum(np.column_stack([green, dead, clover, gdm_blended, total_blended]), 0.0)


In [ ]:
# 4. Dual-Stream Dataset & Focal-Length Augmentation
def apply_camera_scale_simulation(image_np, prob=0.2):
    if random.random() < prob:
        h, w = image_np.shape[:2]
        background = np.zeros_like(image_np)
        scale = random.uniform(0.85, 1.0)
        new_w, new_h = max(1, int(w * scale)), max(1, int(h * scale))
        resized = cv2.resize(image_np, (new_w, new_h), interpolation=cv2.INTER_CUBIC)
        top = random.randint(0, h - new_h)
        left = random.randint(0, w - new_w)
        background[top:top + new_h, left:left + new_w] = resized
        return background
    return image_np

def get_dual_stream_transforms(img_size=512, is_training=True):
    if is_training:
        return transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomApply([transforms.RandomRotation((90, 90))], p=0.5),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean=CFG.IMAGENET_MEAN, std=CFG.IMAGENET_STD),
        ])
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=CFG.IMAGENET_MEAN, std=CFG.IMAGENET_STD),
    ])

class DualStreamBiomassDataset(Dataset):
    def __init__(self, df, img_dir=None, img_size=512, is_training=True, camera_scaling_prob=0.2):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.img_size = img_size
        self.is_training = is_training
        self.camera_scaling_prob = camera_scaling_prob
        self.transform = get_dual_stream_transforms(img_size=img_size, is_training=is_training)
        
        self.has_targets = all(c in self.df.columns for c in CFG.TARGET_ORDER)
        if self.has_targets:
            self.targets_reg = self.df[CFG.TARGET_ORDER].values.astype(np.float32)
            self.targets_cls = get_interval_labels(self.targets_reg, CFG.TARGET_ORDER)

    def __len__(self):
        return len(self.df)

    def _resolve_image_path(self, raw_path):
        if os.path.exists(raw_path):
            return raw_path
        fname = os.path.basename(raw_path)
        candidates = [
            os.path.join(CFG.DATA_DIR, raw_path),
            os.path.join(CFG.DATA_DIR, 'train', fname),
            os.path.join(CFG.DATA_DIR, 'test', fname),
            os.path.join(self.img_dir, fname) if self.img_dir else None,
            os.path.join(self.img_dir, raw_path) if self.img_dir else None,
            os.path.join('train', fname),
            os.path.join('test', fname)
        ]
        for c in candidates:
            if c and os.path.exists(c):
                return c
        for root, _, files in os.walk(CFG.DATA_DIR):
            if fname in files:
                return os.path.join(root, fname)
        raise FileNotFoundError(f'Image {fname} not found in {CFG.DATA_DIR}')

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self._resolve_image_path(row['image_path'])
        raw_bgr = cv2.imread(img_path)
        raw_rgb = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)
        
        mid_w = raw_rgb.shape[1] // 2
        left_np = raw_rgb[:, :mid_w].copy()
        right_np = raw_rgb[:, mid_w:].copy()
        
        if self.is_training and self.camera_scaling_prob > 0:
            left_np = apply_camera_scale_simulation(left_np, prob=self.camera_scaling_prob)
            right_np = apply_camera_scale_simulation(right_np, prob=self.camera_scaling_prob)
            
        tensor_l = self.transform(Image.fromarray(left_np))
        tensor_r = self.transform(Image.fromarray(right_np))
        
        item = {
            'image_left': tensor_l,
            'image_right': tensor_r,
            'sample_id': row.get('sample_id', row.get('clean_id', f'sample_{idx}')),
        }
        if self.has_targets:
            item['targets'] = torch.tensor(self.targets_reg[idx], dtype=torch.float32)
            item['targets_cls'] = torch.tensor(self.targets_cls[idx], dtype=torch.long)
        return item


In [ ]:
# 5. DualStreamBiomassModel Architecture
class DualStreamBiomassModel(nn.Module):
    def __init__(self, backbone_name=CFG.BACKBONE, num_targets=5, num_intervals=7, fusion_dim=384, dropout=0.3, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0)
        self.backbone_dim = self.backbone.num_features
        
        num_heads = 8 if self.backbone_dim % 8 == 0 else 4
        self.cross_view_attn = nn.MultiheadAttention(embed_dim=self.backbone_dim, num_heads=num_heads, dropout=0.1, batch_first=True)
        self.attn_norm = nn.LayerNorm(self.backbone_dim)
        
        self.fusion_mlp = nn.Sequential(
            nn.Linear(self.backbone_dim * 2, fusion_dim),
            nn.LayerNorm(fusion_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        self.reg_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(fusion_dim, fusion_dim // 2),
                nn.LayerNorm(fusion_dim // 2),
                nn.GELU(),
                nn.Dropout(dropout * 0.5),
                nn.Linear(fusion_dim // 2, 64),
                nn.GELU(),
                nn.Linear(64, 1)
            ) for _ in range(num_targets)
        ])
        
        self.cls_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(fusion_dim, 128),
                nn.LayerNorm(128),
                nn.GELU(),
                nn.Dropout(dropout * 0.5),
                nn.Linear(128, num_intervals)
            ) for _ in range(num_targets)
        ])

    def extract_features(self, x):
        feats = self.backbone(x)
        return feats.mean(dim=1) if len(feats.shape) == 3 else feats.mean(dim=[2, 3]) if len(feats.shape) == 4 else feats

    def forward(self, img_left, img_right):
        feat_l = self.extract_features(img_left)
        feat_r = self.extract_features(img_right)
        
        tokens = torch.stack([feat_l, feat_r], dim=1)
        attn_out, _ = self.cross_view_attn(tokens, tokens, tokens)
        tokens = self.attn_norm(tokens + attn_out)
        
        fused = self.fusion_mlp(torch.cat([tokens[:, 0], tokens[:, 1]], dim=-1))
        reg_preds = [F.softplus(head(fused)) for head in self.reg_heads]
        cls_preds = [head(fused) for head in self.cls_heads]
        return reg_preds, cls_preds


In [ ]:
# 6. Two-Stage Training & Cross-Validation
def load_and_pivot_data(train_csv_path):
    if not os.path.exists(train_csv_path):
        # Auto-fallback search
        discovered = os.path.join(CFG.DATA_DIR, 'train.csv')
        if os.path.exists(discovered):
            train_csv_path = discovered
        elif os.path.exists('/kaggle/input'):
            for root, _, files in os.walk('/kaggle/input'):
                if 'train.csv' in files:
                    train_csv_path = os.path.join(root, 'train.csv')
                    break
    print(f'Loading data from: {train_csv_path}')
    df = pd.read_csv(train_csv_path)
    if 'target_name' in df.columns:
        df['clean_id'] = df['sample_id'].astype(str).apply(lambda x: x.split('__')[0])
        targets = df.pivot_table(index='clean_id', columns='target_name', values='target', aggfunc='max').reset_index()
        meta = df[['clean_id', 'image_path', 'Sampling_Date', 'State', 'Species']].drop_duplicates(subset=['clean_id']).reset_index(drop=True)
        wide = pd.merge(meta, targets, on='clean_id', how='left')
        wide['sample_id'] = wide['clean_id']
    else:
        wide = df.copy()
        
    for col in CFG.TARGET_ORDER:
        if col not in wide.columns: wide[col] = 0.0
        wide[col] = wide[col].fillna(0.0)
        
    wide['State_Sampling_Date'] = wide['State'].astype(str) + '_' + wide['Sampling_Date'].astype(str)
    wide['State_Species'] = wide['State'].astype(str) + '_' + wide['Species'].astype(str)
    return wide

train_df = load_and_pivot_data(CFG.TRAIN_CSV)
print(f'Loaded {len(train_df)} training samples.')

sgkf = StratifiedGroupKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=42)
criterion = WeightedBiomassLoss().to(DEVICE)
oof_preds = np.zeros((len(train_df), 5), dtype=np.float32)
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(sgkf.split(train_df, train_df['State_Species'], groups=train_df['State_Sampling_Date'])):
    print(f"\n{'='*25} FOLD {fold + 1} / {CFG.N_FOLDS} {'='*25}")
    f_train, f_val = train_df.iloc[train_idx].reset_index(drop=True), train_df.iloc[val_idx].reset_index(drop=True)
    
    train_loader = DataLoader(DualStreamBiomassDataset(f_train, CFG.TRAIN_IMG_DIR, CFG.IMG_SIZE, is_training=True), batch_size=CFG.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(DualStreamBiomassDataset(f_val, CFG.TRAIN_IMG_DIR, CFG.IMG_SIZE, is_training=False), batch_size=CFG.BATCH_SIZE, shuffle=False)
    
    model = DualStreamBiomassModel(CFG.BACKBONE, pretrained=True).to(DEVICE)
    scaler = torch.amp.GradScaler('cuda')
    
    # Stage 1: Freeze Backbone
    print(f'--- Stage 1: Training Heads ({CFG.STAGE1_EPOCHS} epochs) ---')
    for p in model.backbone.parameters(): p.requires_grad = False
    optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
    
    best_r2 = -float('inf')
    best_val_preds = None
    
    for epoch in range(1, CFG.STAGE1_EPOCHS + 1):
        model.train()
        for batch in train_loader:
            img_l, img_r = batch['image_left'].to(DEVICE), batch['image_right'].to(DEVICE)
            t_reg, t_cls = batch['targets'].to(DEVICE), batch['targets_cls'].to(DEVICE)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                r, c = model(img_l, img_r)
                loss, _, _ = criterion(r, c, t_reg, t_cls)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

    # Stage 2: Unfreeze Backbone
    print(f'--- Stage 2: Full Fine-Tuning ({CFG.STAGE2_EPOCHS} epochs) ---')
    for p in model.backbone.parameters(): p.requires_grad = True
    optimizer = AdamW([
        {'params': model.backbone.parameters(), 'lr': CFG.LR * CFG.BACKBONE_LR_FACTOR},
        {'params': [p for n, p in model.named_parameters() if not n.startswith('backbone')], 'lr': CFG.LR}
    ], weight_decay=CFG.WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=CFG.STAGE2_EPOCHS, eta_min=CFG.LR * 0.01)

    for epoch in range(1, CFG.STAGE2_EPOCHS + 1):
        model.train()
        for batch in train_loader:
            img_l, img_r = batch['image_left'].to(DEVICE), batch['image_right'].to(DEVICE)
            t_reg, t_cls = batch['targets'].to(DEVICE), batch['targets_cls'].to(DEVICE)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                r, c = model(img_l, img_r)
                loss, _, _ = criterion(r, c, t_reg, t_cls)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        scheduler.step()
        
        # Validation
        model.eval()
        val_preds_list, val_true_list = [], []
        with torch.no_grad():
            for batch in val_loader:
                img_l, img_r = batch['image_left'].to(DEVICE), batch['image_right'].to(DEVICE)
                r, _ = model(img_l, img_r)
                val_preds_list.append(torch.cat(r, dim=1).cpu().numpy())
                val_true_list.append(batch['targets'].cpu().numpy())
        v_true = np.concatenate(val_true_list, axis=0)
        v_pred_raw = np.concatenate(val_preds_list, axis=0)
        v_pred_post = soft_physics_postprocess(v_pred_raw)
        r2_post = calculate_competition_r2(v_true, v_pred_post)
        
        if r2_post > best_r2:
            best_r2 = r2_post
            best_val_preds = v_pred_post
            torch.save(model.state_dict(), f'best_model_fold{fold+1}.pt')
            
    print(f'Fold {fold+1} Best SoftBlend R2: {best_r2:.4f}')
    oof_preds[val_idx] = best_val_preds
    fold_scores.append(best_r2)

overall_r2 = calculate_competition_r2(train_df[CFG.TARGET_ORDER].values, oof_preds)
print(f'\n>>> FINAL 5-FOLD OOF R2 SCORE: {overall_r2:.4f} <<<')


In [ ]:
# 7. Dual-Stream Inference & Submission Generation
test_csv_path = CFG.TEST_CSV
if not os.path.exists(test_csv_path) and os.path.exists('/kaggle/input'):
    for root, _, files in os.walk('/kaggle/input'):
        if 'test.csv' in files:
            test_csv_path = os.path.join(root, 'test.csv')
            break

print(f'Reading test data from: {test_csv_path}')
test_df_raw = pd.read_csv(test_csv_path)
if 'target_name' in test_df_raw.columns:
    test_df_raw['clean_id'] = test_df_raw['sample_id'].astype(str).apply(lambda x: x.split('__')[0])
    unique_test = test_df_raw[['clean_id', 'image_path']].drop_duplicates().reset_index(drop=True)
else:
    unique_test = test_df_raw.copy()
    if 'clean_id' not in unique_test.columns: unique_test['clean_id'] = unique_test['sample_id']

test_ds = DualStreamBiomassDataset(unique_test, CFG.TEST_IMG_DIR, CFG.IMG_SIZE, is_training=False, camera_scaling_prob=0.0)
test_loader = DataLoader(test_ds, batch_size=CFG.BATCH_SIZE, shuffle=False)

model_checkpoints = sorted(glob.glob('best_model_fold*.pt'))
print(f'Found {len(model_checkpoints)} fold checkpoints for ensemble.')

all_fold_preds = []
for cp in model_checkpoints:
    model = DualStreamBiomassModel(CFG.BACKBONE, pretrained=False).to(DEVICE)
    model.load_state_dict(torch.load(cp, map_location=DEVICE))
    model.eval()
    
    f_preds = []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f'Predicting {cp}', leave=False):
            img_l, img_r = batch['image_left'].to(DEVICE), batch['image_right'].to(DEVICE)
            # TTA: Standard + Horizontal flip
            r1, _ = model(img_l, img_r)
            r2, _ = model(torch.flip(img_r, [3]), torch.flip(img_l, [3]))
            avg_r = [(a + b) * 0.5 for a, b in zip(r1, r2)]
            f_preds.append(torch.cat(avg_r, dim=1).cpu().numpy())
    all_fold_preds.append(np.concatenate(f_preds, axis=0))

avg_raw = np.mean(all_fold_preds, axis=0)
avg_post = soft_physics_postprocess(avg_raw)

clean_ids = [s['sample_id'] for s in test_ds]
pred_dict = {
    clean_ids[i]: {col: avg_post[i, c_idx] for c_idx, col in enumerate(CFG.TARGET_ORDER)}
    for i in range(len(clean_ids))
}

submission_df = test_df_raw.copy()
submission_df['target'] = submission_df.apply(
    lambda r: pred_dict.get(r['clean_id'], {}).get(r['target_name'], 0.0), axis=1
)
sub = submission_df[['sample_id', 'target']]
sub.to_csv('submission.csv', index=False)
print('Submission saved to submission.csv. Preview:')
print(sub.head(10))
